In [14]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer



print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
DATASET_PATH = "../DATA_SET/Crop recommendation dataset.csv"

df = pd.read_csv(DATASET_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

Dataset loaded successfully.
Shape: (57000, 23)


In [3]:
df.columns = df.columns.str.strip()

print(df.columns.tolist())

['CROPS', 'TYPE_OF_CROP', 'SOIL', 'SEASON', 'SOWN', 'HARVESTED', 'WATER_SOURCE', 'SOIL_PH', 'SOIL_PH_HIGH', 'CROPDURATION', 'CROPDURATION_MAX', 'TEMP', 'MAX_TEMP', 'WATERREQUIRED', 'WATERREQUIRED_MAX', 'RELATIVE_HUMIDITY', 'RELATIVE_HUMIDITY_MAX', 'N', 'N_MAX', 'P', 'P_MAX', 'K', 'K_MAX']


In [4]:
categorical_columns = [
    "CROPS",
    "TYPE_OF_CROP",
    "SOIL",
    "SEASON",
    "SOWN",
    "HARVESTED",
    "WATER_SOURCE"
]

for col in categorical_columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace("\u00a0", " ", regex=False)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

print("Categorical values cleaned successfully.")

Categorical values cleaned successfully.


In [5]:
df["SOIL"] = df["SOIL"].str.lower()

print("Cleaned soil types:")
print(df["SOIL"].unique())

Cleaned soil types:
<StringArray>
[                     'alluvial soil',                         'loamy soil',
                          'clay soil',                  'well-drained soil',
                           'red soil',                    'clay loamy soil',
                   'sandy loamy soil',                         'black soil',
                         'sandy soil',                 'shallow black soil',
                  'black cotton soil',                        'cotton soil',
                  'medium black soil',                   'heavy black soil',
                          'light soi',                         'heavy soil',
                          'deep soil',              'sandy clay loamy soil',
                   'silty loamy soil',              'salty clay loamy soil',
                     'red loamy soil',                   'brown loamy soil',
                      'laterite soil',            'well-drained loamy soil',
                   'light loamy soil',    

In [6]:
print("MISSING VALUES")
print("=" * 60)

missing = df.isnull().sum()

print(missing)

print("\nTotal missing values:", missing.sum())

MISSING VALUES
CROPS                    0
TYPE_OF_CROP             0
SOIL                     0
SEASON                   0
SOWN                     0
HARVESTED                0
WATER_SOURCE             0
SOIL_PH                  0
SOIL_PH_HIGH             0
CROPDURATION             0
CROPDURATION_MAX         0
TEMP                     0
MAX_TEMP                 0
WATERREQUIRED            0
WATERREQUIRED_MAX        0
RELATIVE_HUMIDITY        0
RELATIVE_HUMIDITY_MAX    0
N                        0
N_MAX                    0
P                        0
P_MAX                    0
K                        0
K_MAX                    0
dtype: int64

Total missing values: 0


In [7]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


In [8]:
FEATURES = [
    "SOIL",
    "SEASON",
    "WATER_SOURCE",
    "SOIL_PH",
    "TEMP",
    "RELATIVE_HUMIDITY",
    "N",
    "P",
    "K"
]

TARGET = "CROPS"

X = df[FEATURES].copy()
y = df[TARGET].copy()

print("FEATURES")
print("=" * 50)

for feature in FEATURES:
    print(feature)

print("\nTARGET:", TARGET)

print("\nX shape:", X.shape)
print("y shape:", y.shape)

FEATURES
SOIL
SEASON
WATER_SOURCE
SOIL_PH
TEMP
RELATIVE_HUMIDITY
N
P
K

TARGET: CROPS

X shape: (57000, 9)
y shape: (57000,)


In [9]:
print("CROP DISTRIBUTION")
print("=" * 60)

crop_distribution = y.value_counts()

print(crop_distribution)
print("\nNumber of crop classes:", y.nunique())

CROP DISTRIBUTION
CROPS
rice                 1000
wheat                1000
maize                1000
sorghum              1000
Pearl millet         1000
ragi                 1000
panivaragu           1000
samai                1000
thinai               1000
varagu               1000
kudiraivali          1000
blackgram            1000
greengram            1000
cowpea               1000
bengalgram           1000
horsegram            1000
redgram              1000
soyabean             1000
groundnut            1000
sunflower            1000
gingely              1000
castor               1000
cotton               1000
jute                 1000
sugarcane            1000
sugarbeet            1000
tomato               1000
onion                1000
chillies             1000
Cabbage              1000
bhendi               1000
brinjal              1000
capsicum             1000
pumpkin              1000
snake gourd          1000
ribbed gourd         1000
bottle gourd         1000
bitter gourd  

In [10]:
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Crop classes:")
print("=" * 60)

for index, crop in enumerate(label_encoder.classes_):
    print(f"{index}: {crop}")

print("\nNumber of classes:", len(label_encoder.classes_))

Crop classes:
0: Cabbage
1: Pearl millet
2: annual moringa
3: ash gourd
4: beetroot
5: bengalgram
6: bhendi
7: bitter gourd
8: blackgram
9: bottle gourd
10: brinjal
11: capsicum
12: carrot
13: castor
14: cauliflower
15: chillies
16: chowchow
17: cluster bean
18: cotton
19: cowpea
20: cucumber
21: elephant foot yam
22: french bean
23: gingely
24: greengram
25: groundnut
26: horsegram
27: jute
28: kudiraivali
29: maize
30: muskmelon
31: onion
32: panivaragu
33: peas
34: pumpkin
35: radish
36: ragi
37: redgram
38: ribbed gourd
39: rice
40: samai
41: small onion
42: snake gourd
43: sorghum
44: soyabean
45: sugarbeet
46: sugarcane
47: sunflower
48: sweet potato
49: tapoica
50: thinai
51: tinda
52: tomato
53: varagu
54: vegetable cowpea
55: watermelon
56: wheat

Number of classes: 57


In [11]:
categorical_features = [
    "SOIL",
    "SEASON",
    "WATER_SOURCE"
]

numerical_features = [
    "SOIL_PH",
    "TEMP",
    "RELATIVE_HUMIDITY",
    "N",
    "P",
    "K"
]

print("Categorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)

Categorical features:
['SOIL', 'SEASON', 'WATER_SOURCE']

Numerical features:
['SOIL_PH', 'TEMP', 'RELATIVE_HUMIDITY', 'N', 'P', 'K']


In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("TRAINING DATA")
print("=" * 50)
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nTESTING DATA")
print("=" * 50)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

TRAINING DATA
X_train: (45600, 9)
y_train: (45600,)

TESTING DATA
X_test: (11400, 9)
y_test: (11400,)


In [15]:

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

# Numerical preprocessing
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Complete preprocessing
preprocessor = ColumnTransformer([
    ("categorical", categorical_pipeline, categorical_features),
    ("numerical", numerical_pipeline, numerical_features)
])

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


In [16]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("Preprocessing completed.")

print("Original training shape :", X_train.shape)
print("Processed training shape:", X_train_processed.shape)

print("Original testing shape  :", X_test.shape)
print("Processed testing shape :", X_test_processed.shape)

Preprocessing completed.
Original training shape : (45600, 9)
Processed training shape: (45600, 40)
Original testing shape  : (11400, 9)
Processed testing shape : (11400, 40)


In [17]:
print("Processed data type:", type(X_train_processed))
print("Processed data shape:", X_train_processed.shape)

print("\nFirst 5 processed rows:")
print(X_train_processed[:5])

Processed data type: <class 'numpy.ndarray'>
Processed data shape: (45600, 40)

First 5 processed rows:
[[ 0.          0.          0.          0.          1.          0.
   0.          0.          0.          0.          0.          0.
   0.          0.          0.          0.          0.          0.
   0.          0.          0.          0.          0.          0.
   0.          0.          0.          0.          0.          0.
   1.          0.          1.          0.          1.81005348 -0.60683481
   0.92836398 -0.36656334 -1.0488883  -1.17030993]
 [ 0.          0.          0.          0.          0.          0.
   0.          0.          0.          0.          0.          0.
   0.          0.          1.          0.          0.          0.
   0.          0.          0.          0.          0.          0.
   0.          0.          0.          0.          0.          1.
   0.          0.          0.          1.         -0.49808562 -0.16325694
   0.55264167  1.16523041  1.01583488

In [18]:
feature_names = preprocessor.get_feature_names_out()

print("TOTAL PROCESSED FEATURES:", len(feature_names))
print("=" * 70)

for i, feature in enumerate(feature_names, 1):
    print(f"{i}. {feature}")

TOTAL PROCESSED FEATURES: 40
1. categorical__SOIL_alluvial soil
2. categorical__SOIL_black cotton soil
3. categorical__SOIL_black soil
4. categorical__SOIL_brown loamy soil
5. categorical__SOIL_clay loamy soil
6. categorical__SOIL_clay soil
7. categorical__SOIL_cotton soil
8. categorical__SOIL_deep soil
9. categorical__SOIL_friable soil
10. categorical__SOIL_heavy black soil
11. categorical__SOIL_heavy soil
12. categorical__SOIL_laterite soil
13. categorical__SOIL_light loamy soil
14. categorical__SOIL_light soi
15. categorical__SOIL_loamy soil
16. categorical__SOIL_medium black soil
17. categorical__SOIL_red lateritic loamy soil
18. categorical__SOIL_red loamy soil
19. categorical__SOIL_red soil
20. categorical__SOIL_rich red loamy soil
21. categorical__SOIL_salty clay loamy soil
22. categorical__SOIL_sandy clay loamy soil
23. categorical__SOIL_sandy loamy soil
24. categorical__SOIL_sandy soil
25. categorical__SOIL_shallow black soil
26. categorical__SOIL_silty loamy soil
27. categori

In [19]:
train_class_counts = pd.Series(y_train).value_counts().sort_index()

print("TRAINING CLASS DISTRIBUTION")
print("=" * 60)

for class_id, count in train_class_counts.items():
    crop_name = label_encoder.inverse_transform([class_id])[0]
    print(f"{class_id:2d} | {crop_name:25s} | {count}")

TRAINING CLASS DISTRIBUTION
 0 | Cabbage                   | 800
 1 | Pearl millet              | 800
 2 | annual moringa            | 800
 3 | ash gourd                 | 800
 4 | beetroot                  | 800
 5 | bengalgram                | 800
 6 | bhendi                    | 800
 7 | bitter gourd              | 800
 8 | blackgram                 | 800
 9 | bottle gourd              | 800
10 | brinjal                   | 800
11 | capsicum                  | 800
12 | carrot                    | 800
13 | castor                    | 800
14 | cauliflower               | 800
15 | chillies                  | 800
16 | chowchow                  | 800
17 | cluster bean              | 800
18 | cotton                    | 800
19 | cowpea                    | 800
20 | cucumber                  | 800
21 | elephant foot yam         | 800
22 | french bean               | 800
23 | gingely                   | 800
24 | greengram                 | 800
25 | groundnut                 | 800
26 | horse

In [20]:
import os
import joblib

MODEL_DIR = "../MODELS"

os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(
    preprocessor,
    os.path.join(MODEL_DIR, "preprocessor.pkl")
)

joblib.dump(
    label_encoder,
    os.path.join(MODEL_DIR, "label_encoder.pkl")
)

print("Preprocessing files saved successfully.")

print(
    "Preprocessor:",
    os.path.join(MODEL_DIR, "preprocessor.pkl")
)

print(
    "Label Encoder:",
    os.path.join(MODEL_DIR, "label_encoder.pkl")
)

Preprocessing files saved successfully.
Preprocessor: ../MODELS\preprocessor.pkl
Label Encoder: ../MODELS\label_encoder.pkl
